# Indoor views of a building

This page draws the indoor elements of one building in 3D with `osm.in3d`, the accessor that pyrosm3d adds to pyrosm's `OSM` object: all levels, one level, a web page, and the walking network alone. It then draws frames that were read with pyrosm directly.

The building is Maunula-talo, a community centre in Helsinki whose rooms, doors and stairwells are mapped on three levels. The repository holds a small extract of it for its tests, `tests/data/maunula.osm.pbf`. With a PBF file of your own, give `OSM` a `bounding_box` around the building you want.

## Draw all levels

pyrosm reads the file, and `osm.in3d` holds the views of the `OSM` object. `osm.in3d.plot_indoor()` reads the indoor elements, the walking network and the buildings with pyrosm and returns a plotly figure. It starts seen from the south-east, with north away from you. Drag it to turn it, and click a legend entry to hide or show a layer. Hover over an area or a line to see its name and its level.

- Floors are 4 m apart and coloured by the kind of area.
- Walls stand along the outlines of rooms and are cut away at 1.6 m, so that the rooms stay visible from above. A door leaves a gap in the wall.
- Stairwells and lift shafts are columns through the floors that they join.
- The walking network is drawn on its levels, in the colours of `plot_network`.
- Outer walls, doors and shops are hidden at start. A click on their legend entry shows them.

In [1]:
from pyrosm import OSM

osm = OSM("../tests/data/maunula.osm.pbf")
figure = osm.in3d.plot_indoor()
figure.show()

## Draw one level

With `level`, the figure holds only what is on that level. The columns of stairwells and lifts are left out, since they pass through several levels.

In [2]:
osm.in3d.plot_indoor(level=1).show()

## Write a web page

`osm.in3d.to_html` writes the figure as a web page. The page has a button for all levels and one for each level, with the number of areas on that level under it. A click on a room, a line of the network, a marker or the column of a stairwell or a lift shows what the element is, its levels and its OpenStreetMap tags beside the figure. The Legend button hides and shows the legend.

The page also works on a phone. There the level buttons are one row that scrolls sideways, the legend is hidden at start and the tags lie over the figure. One finger turns the view, and two fingers zoom and move it.

By default the page holds plotly's JavaScript, which makes it about 5 MB larger and usable without a network connection. With `include_plotlyjs="cdn"` the page loads the library from the internet.

In [3]:
osm.in3d.to_html("maunula-talo.html", title="Maunula-talo")

PosixPath('maunula-talo.html')

## Draw the walking network alone

`osm.in3d.plot_network()` reads the walking network with `indoor_elements=True` and draws each edge at its levels. The vertical axis counts levels. With `center` and `radius` it draws only what lies near a point, with `route` a path through the network, and with `graph` a networkx or igraph graph of `get_graph(indoor_elements=True)`, such as the one a route was found in.

In this extract the streets and platforms are at street level, and the lines across the corridors of Maunula-talo are on levels 1 and 2.

In [4]:
osm.in3d.plot_network().show()

## Draw frames that you read yourself

The views draw frames that pyrosm reads: the indoor elements with the walking areas, the walking network with `indoor_elements=True`, and the buildings. The functions `pyrosm3d.plot_indoor` and `pyrosm3d.to_html` take these frames, so you can read them yourself, change them and draw the result. `get_indoor_elements` lists the levels of each element in the column `levels`: the lift in the first row has `level=0-2` and is on three levels.

In [5]:
elements = osm.get_indoor_elements(walking_areas=True)
on_several = elements[elements["levels"].str.contains(";", na=False)]
on_several[["id", "osm_type", "indoor", "highway", "level", "levels"]].head()

,id,osm_type,indoor,highway,level,levels
176,1568797319,way,room,elevator,0-2,0;1;2
178,1576010525,way,room,NaN,1;2,1;2
192,1576010636,way,room,NaN,1;2,1;2


A polygon among the elements is drawn as a floor when it has an `indoor` tag or is a platform, a lift area or a walking area, such as a square; other areas that lines of the walking network cross are floors too. A platform or a walking area without a level takes the level of its lines in the walking network, else street level: the bus and tram platforms and the two squares beside Maunula-talo have no `level` tag and are on street level. Walls mapped as lines, doors, and the shops and amenities that have a level are drawn from the same frame.

Here, we draw the second floor from the elements alone, without the walking network.

In [6]:
import pyrosm3d

pyrosm3d.plot_indoor(elements, buildings=osm.get_buildings(), level=2).show()